# Getting started

A field on a uniform grid: create it, fill it, evaluate it, plot it and compute with it.

A `Field` is a grid from simgeometry plus the values on its nodes, one or several components
per node. This page goes through the basic operations on one process; the
[MPI tutorial](/mpifield/tutorials/mpi-and-halos/) splits the same fields over several.

In [ ]:
import numpy as np
import plasma_plots  # noqa: F401  (registers the .plasma accessor on xarray)

from mpifield import Field

## A field on a grid

`num_gridpoints` nodes span `lower_bound` to `upper_bound` along each axis, so the spacing is
`(upper - lower) / (n - 1)`. A field starts at zero; `get_grid_points` gives the node
coordinates of each axis.

In [ ]:
phi = Field(
    ndim=2,
    lower_bound=(0.0, 0.0),
    upper_bound=(0.1, 0.05),
    num_gridpoints=(101, 51),
)
x, y = np.meshgrid(*phi.get_grid_points(), indexing="ij")
print(phi)
print("spacing:", phi.grid_spacing)

`set_field_data` takes a scalar, an array of the grid's shape (for a scalar field or one
component) or of the full shape `num_gridpoints + (num_components,)`.

In [ ]:
phi.set_field_data(1000.0 * np.exp(-((x - 0.05) ** 2 + (y - 0.025) ** 2) / 1e-4))
print(phi.max(), phi.sum())

## Evaluating and plotting

`evaluate` returns an `xarray.DataArray`: the nodes as they are, or interpolated on a line, a
plane or scattered points. A scalar fixes a coordinate. `plot` draws the same with
[plasma-plots](https://github.com/max-models/plasma-plots): a line for one free coordinate,
a map for two.

In [ ]:
profile = phi.evaluate(y=0.025, name="phi", units="V")
profile

In [ ]:
phi.plot(name="phi", units="V", equal_aspect=True);

In [ ]:
phi.plot(y=0.025, name="phi", units="V");

## Computing with fields

Fields combine elementwise with numbers, arrays and other fields on the same grid; NumPy
ufuncs apply too. `gradient` differentiates a scalar field into a vector field with one
component per axis; a scalar field broadcasts over the components of a vector field, so
`E / |E|` is a unit vector field.

In [ ]:
E = -phi.gradient()
magnitude = E.magnitude()
unit = E / (magnitude + 1e-12)
print(E.num_components, magnitude.num_components, float(unit.magnitude().max()))
magnitude.plot(name="|E|", units="V/m", equal_aspect=True);

In [ ]:
E.evaluate(name="E").plasma.plot.vector(x="x", y="y", stride=6);

## Saving

`export` writes a NetCDF file with the values, the bounds and the periodic axes; `load`
reads it back, split over the ranks as asked.

In [ ]:
import tempfile
from pathlib import Path

path = Path(tempfile.mkdtemp()) / "phi.nc"
phi.export(path)
again = Field.load(path)
print(again, np.array_equal(again.to_numpy(), phi.to_numpy()))